In [2]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser


# Load environment variables from .env file
load_dotenv(override=True)

llm_openai = ChatOpenAI(model = "deepseek-chat",
base_url="https://api.deepseek.com",
api_key=os.environ.get('DEEPSEEK_API_KEY'),
 temperature=0)


if os.environ.get('DEEPSEEK_API_KEY') :
    print("DEEPSEEK_API_KEY is set")
else:    print("DEEPSEEK_API_KEY is not set")

llm_openai.invoke("你好").content


DEEPSEEK_API_KEY is set


'你好！很高兴见到你。有什么我可以帮你的吗？无论是聊天、解答问题，还是需要一些建议，我都在这里。😊'

In [3]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

In [4]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [5]:
# search_tool.invoke("北京今天天气?")

In [6]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
model = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com",
    temperature=0,
    timeout=30,
)


In [7]:
from pydantic import BaseModel, Field

class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer.")
    url: str = Field(description="The URL of the web page cited in the answer.")

class AnswerInfo(BaseModel):
    answer: str = Field(description="The final answer for user.")
    references: list[Reference] = Field(description="The web pages cited in the answer.")    

agent = create_agent(model, tools=[search_tool],system_prompt="你必须使用提供的工具回答问题，绝对不能自己回答！")

    

In [10]:
response = agent.invoke({
        "messages": [HumanMessage(content="北京今天天气?")],
    })

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

北京今天天气?
================================== Ai Message ==================================

我来帮你查询北京今天的天气情况。
Tool Calls:
  tavily_search (call_00_HlMpNVHy53QeeugYce5G5723)
 Call ID: call_00_HlMpNVHy53QeeugYce5G5723
  Args:
    query: 北京今天天气 2025年
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "北京今天天气 2025年", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://weather.cma.cn/web/weather/54511.html", "title": "中国气象局-天气预报- 北京", "content": "| 气温 | 19℃ | 15.2℃ | 13℃ | 18.6℃ | 26.1℃ | 26.2℃ | 24.9℃ | 19.9℃ |. | 风速 | 3.2m/s | 2.3m/s | 2m/s | 3.3m/s | 3.1m/s | 3.2m/s | 3.3m/s | 2.3m/s |. | 风向 | 西南风 | 西南风 | 东北风 | 东北风 | 东北风 | 东南风 | 东南风 | 东南风 |. | 湿度 | 57.2% | 64.2% | 71.4% | 59.7% | 32.4% | 31.7% | 37.8% | 32.3% |. | 降水 | 无降水 | 无降水 | 无降水 | 无降水 | 1.3mm | 无降水 | 无降水 | 无降水 |. | 风向 | 西北风 | 西南风 | 西南风